# 36 - QUAL-002 / QUAL-004: Held-out con Ground Truth y evaluación Dice/IoU — versión corregida

Este notebook arma un held-out preliminar con pares `image/mask`, ejecuta `scripts/evaluate_model.py` y guarda evidencia para Claude/Codex.

## Corrección de esta versión

La versión anterior intentaba crear las carpetas `qual/heldout/...` directamente dentro de Google Drive y falló con:

```text
OSError: [Errno 107] Transport endpoint is not connected
```

Eso suele ser un problema temporal del montaje de Drive al copiar archivos dentro de Drive. Esta versión evita ese punto débil:

- copia el held-out a disco local de Colab: `/content/pfi_qual_work/heldout`;
- ejecuta el evaluador contra esas carpetas locales;
- guarda JSON/logs/resúmenes localmente;
- al final sincroniza solo la evidencia chica hacia Drive.

## Advertencia metodológica

Si no se confirma que los casos seleccionados nunca estuvieron en entrenamiento, reportar como **evaluación preliminar con ground truth**, no como test limpio final por paciente.

In [ ]:
# ============================================
# 0) Montar Google Drive e instalar dependencias
# ============================================
from google.colab import drive

drive.mount('/content/drive')

!pip -q install SimpleITK pydicom nibabel pandas pillow

In [ ]:
# ============================================
# 1) Configuración principal
# ============================================
from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib, csv, re, time, textwrap
from datetime import datetime, timezone

import numpy as np
import pandas as pd

DRIVE_ROOT = Path("/content/drive/MyDrive")
PFI_ROOT = DRIVE_ROOT / "PFI_MVP"

SAGITTAL_CKPT = PFI_ROOT / "models/final/sagittal_spider_multiclass_final_best.pt"
AXIAL_CKPT    = PFI_ROOT / "models/final/axial_t2_alkafri_final_best.pt"

# Salidas finales en Drive, solo evidencia chica
DRIVE_QUAL_ROOT = PFI_ROOT / "qual"
DRIVE_REPORTS_ROOT = DRIVE_QUAL_ROOT / "reports"
DRIVE_DOCS_ROOT = PFI_ROOT / "docs"

# Workdir local para evitar error Errno 107 copiando Drive -> Drive
LOCAL_WORK_ROOT = Path("/content/pfi_qual_work")
LOCAL_HELDOUT_ROOT = LOCAL_WORK_ROOT / "heldout"
LOCAL_REPORTS_ROOT = LOCAL_WORK_ROOT / "reports"
LOCAL_DOCS_ROOT = LOCAL_WORK_ROOT / "docs"

SAG_TEST_DIR = LOCAL_HELDOUT_ROOT / "sagittal"
AX_TEST_DIR  = LOCAL_HELDOUT_ROOT / "axial"

SAG_IMAGES_OUT = SAG_TEST_DIR / "images"
SAG_MASKS_OUT  = SAG_TEST_DIR / "masks"
AX_IMAGES_OUT  = AX_TEST_DIR / "images"
AX_MASKS_OUT   = AX_TEST_DIR / "masks"

for d in [LOCAL_WORK_ROOT, LOCAL_REPORTS_ROOT, LOCAL_DOCS_ROOT, SAG_IMAGES_OUT, SAG_MASKS_OUT, AX_IMAGES_OUT, AX_MASKS_OUT, DRIVE_REPORTS_ROOT, DRIVE_DOCS_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

CLEAR_EXISTING_LOCAL_HELDOUT = True
TARGET_SIZE = 256

print("PFI_ROOT:", PFI_ROOT, "| exists:", PFI_ROOT.exists())
print("SAGITTAL_CKPT:", SAGITTAL_CKPT, "| exists:", SAGITTAL_CKPT.exists())
print("AXIAL_CKPT:", AXIAL_CKPT, "| exists:", AXIAL_CKPT.exists())
print("LOCAL_WORK_ROOT:", LOCAL_WORK_ROOT)
print("SAG_TEST_DIR:", SAG_TEST_DIR)
print("AX_TEST_DIR:", AX_TEST_DIR)

In [ ]:
# ============================================
# 2) Utilidades comunes + copia robusta
# ============================================
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def reset_dir(path: Path):
    if path.exists() and CLEAR_EXISTING_LOCAL_HELDOUT:
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def remount_drive_if_needed():
    try:
        from google.colab import drive as colab_drive
        print("Intentando force_remount de Google Drive...")
        colab_drive.mount('/content/drive', force_remount=True)
    except Exception as e:
        print("No pude remontar automáticamente:", repr(e))


def copy_file_robust(src: Path, dst: Path, retries: int = 3, sleep_s: float = 2.0):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            shutil.copyfile(src, dst)
            return dst
        except OSError as e:
            last_error = e
            print(f"Copy failed attempt {attempt}/{retries}: {src} -> {dst} | {repr(e)}")
            if getattr(e, 'errno', None) == 107 or "Transport endpoint" in str(e):
                remount_drive_if_needed()
            time.sleep(sleep_s)
        except Exception as e:
            last_error = e
            print(f"Copy failed attempt {attempt}/{retries}: {src} -> {dst} | {repr(e)}")
            time.sleep(sleep_s)
    raise RuntimeError(f"No se pudo copiar {src} -> {dst}. Último error: {repr(last_error)}")


def copy_pair(image_src: Path, mask_src: Path, image_dst: Path, mask_dst: Path):
    copy_file_robust(image_src, image_dst)
    copy_file_robust(mask_src, mask_dst)


def print_tree_counts():
    print("Sagital images:", len(list(SAG_IMAGES_OUT.glob("*"))))
    print("Sagital masks:", len(list(SAG_MASKS_OUT.glob("*"))))
    print("Axial images:", len(list(AX_IMAGES_OUT.glob("*"))))
    print("Axial masks:", len(list(AX_MASKS_OUT.glob("*"))))

## 3) Buscar splits/inventarios existentes

Esta celda intenta encontrar archivos que indiquen split, inventario, overview, train/validation/test, etc.

In [ ]:
# ============================================
# 3) Buscar archivos candidatos de split / inventario
# ============================================
patterns = ["*split*", "*overview*", "*holdout*", "*test*", "*train*", "*validation*", "*val*", "*pairing*", "*inventory*", "*manifest*"]
valid_suffixes = {".csv", ".json", ".txt", ".parquet", ".tsv"}

hits = []
for pat in patterns:
    for p in PFI_ROOT.rglob(pat):
        if p.is_file() and p.suffix.lower() in valid_suffixes:
            normalized = str(p).replace("\\", "/").lower()
            if "/qual/heldout/" in normalized:
                continue
            hits.append(p)

hits = sorted(set(hits), key=lambda p: str(p).lower())
print("Archivos candidatos de split/inventario:", len(hits))
for i, p in enumerate(hits[:250]):
    print(f"{i:03d} | {p}")

split_scan_path = LOCAL_REPORTS_ROOT / "qual-002-split-candidates.txt"
split_scan_path.write_text("\n".join(str(p) for p in hits), encoding="utf-8")
print("\nGuardado local:", split_scan_path)

In [ ]:
# ============================================
# 3b) Preview automático de CSV/TSV/JSON chicos
# ============================================
def preview_table_file(path: Path, max_rows=5):
    print("\n" + "=" * 100)
    print(path)
    try:
        if path.suffix.lower() in [".csv", ".tsv"]:
            sep = "\t" if path.suffix.lower() == ".tsv" else ","
            df = pd.read_csv(path, sep=sep)
            print("shape:", df.shape)
            print("columns:", list(df.columns)[:30])
            for col in df.columns:
                lc = col.lower()
                if lc in ["split", "subset", "fold", "set", "phase", "partition"]:
                    print(f"value_counts[{col}]:")
                    print(df[col].value_counts(dropna=False).head(20))
            print(df.head(max_rows).to_string(index=False))
        elif path.suffix.lower() == ".json":
            text = path.read_text(encoding="utf-8", errors="ignore")
            obj = json.loads(text)
            if isinstance(obj, dict):
                print("json keys:", list(obj.keys())[:30])
            elif isinstance(obj, list):
                print("json list len:", len(obj))
                print("first item:", obj[0] if obj else None)
    except Exception as e:
        print("No pude previsualizar:", repr(e))

for p in hits[:25]:
    preview_table_file(p)

## 4) Verificar pares sagitales SPIDER

In [ ]:
# ============================================
# 4) Pares sagitales SPIDER
# ============================================
SAG_IMAGES_SRC = PFI_ROOT / "data/SPIDER/images/images"
SAG_MASKS_SRC = PFI_ROOT / "data/SPIDER/masks/masks"

print("SAG_IMAGES_SRC:", SAG_IMAGES_SRC, "| exists:", SAG_IMAGES_SRC.exists())
print("SAG_MASKS_SRC:", SAG_MASKS_SRC, "| exists:", SAG_MASKS_SRC.exists())

def find_sag_mask_for_image(img: Path):
    exact = SAG_MASKS_SRC / img.name
    if exact.exists():
        return exact
    candidates = [p for p in SAG_MASKS_SRC.glob(img.stem + "*") if p.is_file()]
    if candidates:
        return sorted(candidates, key=lambda p: str(p))[0]
    case_id = img.name.split("_")[0]
    candidates = [p for p in SAG_MASKS_SRC.glob(case_id + "*") if p.is_file()]
    if candidates:
        return sorted(candidates, key=lambda p: str(p))[0]
    return None

sag_pairs = []
missing_sag_masks = []
for img in sorted(SAG_IMAGES_SRC.glob("*.mha")):
    if "_SPACE" in img.name:
        continue
    mask = find_sag_mask_for_image(img)
    if mask is not None and mask.exists():
        sag_pairs.append((img, mask))
    else:
        missing_sag_masks.append(img)

print("Pares sagitales encontrados:", len(sag_pairs))
print("Imágenes sagitales sin máscara detectada:", len(missing_sag_masks))
for img, mask in sag_pairs[:20]:
    print(img.name, "|", mask.name)

## 5) Verificar pares axiales Al-Kafri/Sudirman

In [ ]:
# ============================================
# 5) Pares axiales procesados
# ============================================
AXIAL_PAIRING_SRC = PFI_ROOT / "data/AXIAL_ALKAFRI/processed/pairing_v1"
print("AXIAL_PAIRING_SRC:", AXIAL_PAIRING_SRC, "| exists:", AXIAL_PAIRING_SRC.exists())

axial_pairs = []
for img in sorted(AXIAL_PAIRING_SRC.glob("*_image.npy")):
    mask = AXIAL_PAIRING_SRC / img.name.replace("_image.npy", "_mask.npy")
    if mask.exists():
        pair_id = img.name.replace("_image.npy", "")
        axial_pairs.append((pair_id, img, mask))

print("Pares axiales encontrados:", len(axial_pairs))
for pair_id, img, mask in axial_pairs[:20]:
    print(pair_id, "|", img.name, "|", mask.name)

if axial_pairs:
    arr = np.load(axial_pairs[0][1])
    msk = np.load(axial_pairs[0][2])
    print("\nPrimer axial image shape/dtype/range:", arr.shape, arr.dtype, float(np.nanmin(arr)), float(np.nanmax(arr)))
    print("Primer axial mask shape/dtype/range:", msk.shape, msk.dtype, float(np.nanmin(msk)), float(np.nanmax(msk)))

## 6) Selección del held-out

Por defecto usa 10 casos sagitales T2 y 30 pares axiales. Cambiá `USER_SAG_CASE_NAMES` / `USER_AXIAL_PAIR_IDS` si encontrás un split limpio.

In [ ]:
# ============================================
# 6) Selección del held-out
# ============================================
SELECTION_MODE = "preliminary_gt_not_confirmed_unseen"  # cambiar a clean_test_from_split si corresponde
USER_SAG_CASE_NAMES = None
USER_AXIAL_PAIR_IDS = None

DEFAULT_SAG_CASE_NAMES = [
    "101_t2.mha", "104_t2.mha", "105_t2.mha", "106_t2.mha", "107_t2.mha",
    "109_t2.mha", "113_t2.mha", "115_t2.mha", "116_t2.mha", "118_t2.mha",
]
DEFAULT_AXIAL_PAIR_IDS = [f"pair_{i:04d}" for i in range(1, 31)]

sag_case_names = USER_SAG_CASE_NAMES or DEFAULT_SAG_CASE_NAMES
axial_pair_ids = USER_AXIAL_PAIR_IDS or DEFAULT_AXIAL_PAIR_IDS

available_sag_by_name = {img.name: (img, mask) for img, mask in sag_pairs}
selected_sag = []
for name in sag_case_names:
    if name in available_sag_by_name:
        selected_sag.append((name, *available_sag_by_name[name]))
if len(selected_sag) < len(sag_case_names):
    already = {name for name, _, _ in selected_sag}
    for img, mask in sag_pairs:
        if img.name not in already:
            selected_sag.append((img.name, img, mask))
        if len(selected_sag) >= len(sag_case_names):
            break

available_ax_by_id = {pair_id: (img, mask) for pair_id, img, mask in axial_pairs}
selected_axial = []
for pair_id in axial_pair_ids:
    if pair_id in available_ax_by_id:
        selected_axial.append((pair_id, *available_ax_by_id[pair_id]))
if len(selected_axial) < len(axial_pair_ids):
    already = {pair_id for pair_id, _, _ in selected_axial}
    for pair_id, img, mask in axial_pairs:
        if pair_id not in already:
            selected_axial.append((pair_id, img, mask))
        if len(selected_axial) >= len(axial_pair_ids):
            break

print("SELECTION_MODE:", SELECTION_MODE)
print("Selected sagittal:", len(selected_sag))
for item in selected_sag:
    print(" -", item[0], "|", item[1], "|", item[2])
print("\nSelected axial:", len(selected_axial))
for item in selected_axial[:40]:
    print(" -", item[0], "|", item[1].name, "|", item[2].name)

if len(selected_sag) == 0:
    raise RuntimeError("No se seleccionó ningún caso sagital. Revisar rutas/máscaras SPIDER.")
if len(selected_axial) == 0:
    raise RuntimeError("No se seleccionó ningún caso axial. Revisar pairing_v1.")

## 7) Crear carpetas held-out locales `images/` + `masks/`

Esta es la celda corregida: copia desde Drive hacia `/content/pfi_qual_work/heldout`, no hacia Drive.

In [ ]:
# ============================================
# 7) Crear carpetas held-out locales images/ + masks/
# ============================================
reset_dir(SAG_IMAGES_OUT)
reset_dir(SAG_MASKS_OUT)
reset_dir(AX_IMAGES_OUT)
reset_dir(AX_MASKS_OUT)

manifest_rows = []

for case_id, img, mask in selected_sag:
    img_dst = SAG_IMAGES_OUT / img.name
    mask_dst = SAG_MASKS_OUT / mask.name
    copy_pair(img, mask, img_dst, mask_dst)
    manifest_rows.append({
        "plane": "sagittal",
        "case_id": case_id,
        "image_source": str(img),
        "mask_source": str(mask),
        "image_dest_local": str(img_dst),
        "mask_dest_local": str(mask_dst),
        "selection_mode": SELECTION_MODE,
    })

for pair_id, img, mask in selected_axial:
    img_dst = AX_IMAGES_OUT / f"{pair_id}.npy"
    mask_dst = AX_MASKS_OUT / f"{pair_id}.npy"
    copy_pair(img, mask, img_dst, mask_dst)
    manifest_rows.append({
        "plane": "axial",
        "case_id": pair_id,
        "image_source": str(img),
        "mask_source": str(mask),
        "image_dest_local": str(img_dst),
        "mask_dest_local": str(mask_dst),
        "selection_mode": SELECTION_MODE,
    })

print_tree_counts()
manifest_df = pd.DataFrame(manifest_rows)
manifest_csv = LOCAL_REPORTS_ROOT / "qual-002-heldout-manifest.csv"
manifest_json = LOCAL_REPORTS_ROOT / "qual-002-heldout-manifest.json"
manifest_df.to_csv(manifest_csv, index=False)
manifest_json.write_text(json.dumps({
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "selection_mode": SELECTION_MODE,
    "warning": "Si selection_mode != clean_test_from_split, reportar como evaluación preliminar con ground truth, no como test limpio final.",
    "sagittal_test_dir_local": str(SAG_TEST_DIR),
    "axial_test_dir_local": str(AX_TEST_DIR),
    "counts": {"sagittal": len(selected_sag), "axial": len(selected_axial)},
    "rows": manifest_rows,
}, indent=2, ensure_ascii=False), encoding="utf-8")
print("Manifest CSV local:", manifest_csv)
print("Manifest JSON local:", manifest_json)
manifest_df.head()

## 8) Ubicar repo y `evaluate_model.py`

In [ ]:
# ============================================
# 8) Ubicar repo y evaluator
# ============================================
REPO_CANDIDATES = [
    PFI_ROOT / "repo",
    PFI_ROOT / "PFI_MVPTest_Enzo_AImodule",
    DRIVE_ROOT / "PFI_MVPTest_Enzo_AImodule",
    Path("/content/PFI_MVPTest_Enzo_AImodule"),
    Path.cwd(),
]
found = []
for repo in REPO_CANDIDATES:
    script = repo / "scripts/evaluate_model.py"
    if script.exists():
        found.append((repo, script))
if not found:
    for script in PFI_ROOT.rglob("evaluate_model.py"):
        if script.name == "evaluate_model.py":
            found.append((script.parent.parent, script))

print("Evaluators encontrados:", len(found))
for i, (repo, script) in enumerate(found):
    print(f"{i:02d} | repo={repo} | script={script}")
if not found:
    raise RuntimeError("No encontré scripts/evaluate_model.py. Subí/cloná el repo o ajustá REPO_CANDIDATES.")

REPO_ROOT, EVALUATE_SCRIPT = found[0]
print("\nUsando REPO_ROOT:", REPO_ROOT)
print("Usando EVALUATE_SCRIPT:", EVALUATE_SCRIPT)

pythonpath_candidates = []
if (REPO_ROOT / "ai_service").exists():
    pythonpath_candidates.append(REPO_ROOT / "ai_service")
pythonpath_candidates.append(REPO_ROOT)
os.environ["PYTHONPATH"] = str(pythonpath_candidates[0])
print("PYTHONPATH:", os.environ["PYTHONPATH"])

## 9) Ejecutar QUAL-004

In [ ]:
# ============================================
# 9) Ejecutar evaluate_model.py
# ============================================
def run_eval(plane: str, checkpoint: Path, test_dir: Path, num_classes: int, output_json: Path):
    log_path = output_json.with_suffix(".log")
    cmd = [
        sys.executable, str(EVALUATE_SCRIPT),
        "--plane", plane,
        "--checkpoint", str(checkpoint),
        "--test-dir", str(test_dir),
        "--num-classes", str(num_classes),
        "--target-size", str(TARGET_SIZE),
        "--output", str(output_json),
    ]
    env = os.environ.copy()
    env["PYTHONPATH"] = os.environ.get("PYTHONPATH", "")
    print("\n" + "=" * 100)
    print("Running:", " ".join(cmd))
    print("cwd:", REPO_ROOT)
    print("PYTHONPATH:", env["PYTHONPATH"])
    proc = subprocess.run(cmd, cwd=str(REPO_ROOT), env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    log_text = proc.stdout
    log_path.write_text(log_text, encoding="utf-8", errors="ignore")
    print(log_text[-5000:])
    print("returncode:", proc.returncode)
    print("log:", log_path)
    print("output exists:", output_json.exists(), output_json)
    if proc.returncode != 0:
        raise RuntimeError(f"Falló evaluación {plane}. Ver log: {log_path}")
    return output_json, log_path

sag_out_json = LOCAL_DOCS_ROOT / "qual-004-sagittal.json"
ax_out_json  = LOCAL_DOCS_ROOT / "qual-004-axial.json"

sag_json, sag_log = run_eval("sagittal", SAGITTAL_CKPT, SAG_TEST_DIR, 4, sag_out_json)
ax_json, ax_log = run_eval("axial", AXIAL_CKPT, AX_TEST_DIR, 6, ax_out_json)

print("\nEvaluación finalizada.")
print("Sagital:", sag_json)
print("Axial:", ax_json)

## 10) Leer JSONs y generar resumen para Claude/Codex

In [ ]:
# ============================================
# 10) Mostrar resultados y generar resumen
# ============================================
def load_json(path: Path):
    if not path.exists():
        print("NO EXISTE:", path)
        return None
    return json.loads(path.read_text(encoding="utf-8"))

def find_metric_recursive(obj, possible_keys):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in possible_keys:
                return v
        for v in obj.values():
            found = find_metric_recursive(v, possible_keys)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for item in obj:
            found = find_metric_recursive(item, possible_keys)
            if found is not None:
                return found
    return None

def pretty_json(obj):
    return json.dumps(obj, indent=2, ensure_ascii=False, default=str)

sag_result = load_json(sag_out_json)
ax_result = load_json(ax_out_json)

print("\n==== qual-004-sagittal.json ====")
print(pretty_json(sag_result)[:12000] if sag_result is not None else "NO JSON")
print("\n==== qual-004-axial.json ====")
print(pretty_json(ax_result)[:12000] if ax_result is not None else "NO JSON")

metric_keys = {"diceMacroForeground", "dice_macro_foreground", "macro_dice_foreground", "diceMacroNoBackground", "dice_macro_no_bg", "dice_macro_no_bg_excluding_background"}
sag_macro = find_metric_recursive(sag_result, metric_keys) if sag_result is not None else None
ax_macro = find_metric_recursive(ax_result, metric_keys) if ax_result is not None else None

summary_md = f'''# QUAL-004 - Evaluación preliminar con Ground Truth

Fecha UTC: {datetime.now(timezone.utc).isoformat()}

## Advertencia metodológica

selection_mode: `{SELECTION_MODE}`

Si `selection_mode` no es `clean_test_from_split`, estos números deben reportarse como **evaluación preliminar con ground truth**, no como test limpio final por paciente.

## Inputs

Sagital:
- test_dir local: `{SAG_TEST_DIR}`
- checkpoint: `{SAGITTAL_CKPT}`
- casos: {len(selected_sag)}

Axial:
- test_dir local: `{AX_TEST_DIR}`
- checkpoint: `{AXIAL_CKPT}`
- casos: {len(selected_axial)}

## Outputs locales

- `{sag_out_json}`
- `{ax_out_json}`
- `{sag_log}`
- `{ax_log}`
- `{manifest_json}`

## Métricas principales detectadas

- Sagital dice macro foreground/no-bg: `{sag_macro}`
- Axial dice macro foreground/no-bg: `{ax_macro}`

## Nota para Claude/Codex

Los JSON completos están guardados en `qual-004-sagittal.json` y `qual-004-axial.json`. Usar esos archivos como fuente de verdad para QUAL-005 / planificación de mejora de entrenamiento.
'''.strip()

summary_path = LOCAL_DOCS_ROOT / "qual-004-summary.md"
summary_path.write_text(summary_md, encoding="utf-8")
print("\n==== SUMMARY PARA CLAUDE/CODEX ====")
print(summary_md)
print("\nGuardado local:", summary_path)

## 11) Sincronizar evidencia a Drive y comprimir

Solo se copian archivos chicos: JSONs, logs, manifest, summary y zip de evidencia.

In [ ]:
# ============================================
# 11) Sincronizar evidencia a Drive y comprimir
# ============================================
import zipfile

files_to_sync = [
    LOCAL_REPORTS_ROOT / "qual-002-split-candidates.txt",
    LOCAL_REPORTS_ROOT / "qual-002-heldout-manifest.csv",
    LOCAL_REPORTS_ROOT / "qual-002-heldout-manifest.json",
    LOCAL_DOCS_ROOT / "qual-004-sagittal.json",
    LOCAL_DOCS_ROOT / "qual-004-axial.json",
    LOCAL_DOCS_ROOT / "qual-004-sagittal.log",
    LOCAL_DOCS_ROOT / "qual-004-axial.log",
    LOCAL_DOCS_ROOT / "qual-004-summary.md",
]

synced = []
for p in files_to_sync:
    if not p.exists():
        print("No existe local, no se sincroniza:", p)
        continue
    dst = (DRIVE_REPORTS_ROOT if p.parent == LOCAL_REPORTS_ROOT else DRIVE_DOCS_ROOT) / p.name
    copy_file_robust(p, dst)
    synced.append(dst)
    print("Sincronizado:", p, "->", dst)

local_zip_path = LOCAL_REPORTS_ROOT / "qual-002-004-evidence.zip"
zip_path = DRIVE_REPORTS_ROOT / "qual-002-004-evidence.zip"
with zipfile.ZipFile(local_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in files_to_sync:
        if p.exists():
            z.write(p, arcname=p.name)
copy_file_robust(local_zip_path, zip_path)

print("ZIP local:", local_zip_path)
print("ZIP Drive:", zip_path)
print("ZIP exists:", zip_path.exists())
if zip_path.exists():
    print("ZIP size MB:", round(zip_path.stat().st_size / (1024 * 1024), 3))
    print("ZIP sha256:", sha256_file(zip_path))

## 12) Mensaje sugerido para Claude/Codex

In [ ]:
print(f'''
QUAL-002 / QUAL-004 ejecutado en Colab.

Held-out:
- selection_mode: {SELECTION_MODE}
- sagital casos: {len(selected_sag)}
- axial casos: {len(selected_axial)}

Carpetas usadas por el evaluador:
- sagital test_dir local: {SAG_TEST_DIR}
- axial test_dir local: {AX_TEST_DIR}

Outputs en Drive:
- {DRIVE_DOCS_ROOT / "qual-004-sagittal.json"}
- {DRIVE_DOCS_ROOT / "qual-004-axial.json"}
- {DRIVE_DOCS_ROOT / "qual-004-summary.md"}
- {DRIVE_REPORTS_ROOT / "qual-002-heldout-manifest.json"}
- {DRIVE_REPORTS_ROOT / "qual-002-004-evidence.zip"}

Aclaración:
Si selection_mode != clean_test_from_split, reportar como evaluación preliminar con GT,
no como test limpio final por paciente.

Copio a continuación el summary y/o los dos JSON.
''')